# Mortality feature comparison: elastic-net Cox, XGBoost-Cox, and RSF

Re-run the existing elastic-net Cox feature-comparison analysis and extend it to the other two survival algorithms for mortality only. Each model contains the shared baseline variables plus exactly one modality: stage, first-line treatment, labs, somatic alterations, PRS, or text embeddings.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'python_scripts' / 'model_training' / 'slurm_array_utils.py').exists():
            return candidate
    raise FileNotFoundError('Could not locate the clinical_text_embedding_project repository root.')

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / 'jupyter_notebooks' / 'mortality_model_comparison'
sys.path.insert(0, str(NOTEBOOK_DIR))
sys.path.insert(0, str(REPO_ROOT / 'python_scripts' / 'model_training'))

from slurm_array_utils import (
    SURV_PATH,
    filter_event_rows,
    load_feature_modalities_df,
)
from survival_benchmark import (
    DEFAULT_PARAM_GRIDS,
    create_train_test_split,
    generate_oof_risk_scores,
    run_train_test_benchmark,
)

pd.set_option('display.max_colwidth', 100)

## Configuration

This notebook runs 18 comparisons: three algorithms × six modalities. Hyperparameters are selected independently for each comparison using five folds of the shared 80% training set.

In [ ]:
RANDOM_STATE = 1234
N_JOBS = int(os.getenv('SLURM_CPUS_PER_TASK', '1'))
MODELS = ('elastic_net_cox', 'xgboost_cox', 'rsf')
MODALITIES = ('stage', 'treatment', 'labs', 'somatic', 'prs', 'text')
PARAM_GRIDS = {name: DEFAULT_PARAM_GRIDS[name] for name in MODELS}
OUTPUT_DIR = (
    Path(SURV_PATH)
    / 'results'
    / 'death_met_results'
    / 'mortality_feature_comparison_all_models'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## Build the shared feature-comparison cohort

`load_feature_modalities_df(..., modality=None)` inner-merges all six modality sources, reproducing the existing common feature-file intersection. Raw lab measurements with a paired missingness indicator are retained for fold-specific mean imputation. Rows missing any other required value are removed once, before splitting, so every comparison uses identical patients.

In [ ]:
full_df, cancer_type_cols, _, modality_cfg, _ = load_feature_modalities_df(
    'death_met', modality=None
)
cohort = filter_event_rows(full_df, 'death')
baseline_cols = ['GENDER', 'AGE_AT_TREATMENTSTART'] + cancer_type_cols

def unique_in_order(values):
    return list(dict.fromkeys(values))

feature_sets = {
    modality: unique_in_order(baseline_cols + modality_cfg[modality]['penalized_cols'])
    for modality in MODALITIES
}
all_feature_cols = unique_in_order(
    col for modality in MODALITIES for col in feature_sets[modality]
)
required_nonfeature = ['DFCI_MRN', 'death', 'tt_death']
imputable_lab_cols = {
    col for col in all_feature_cols if f'{col}_missing' in cohort.columns
}
drop_subset = required_nonfeature + [
    col for col in all_feature_cols if col not in imputable_lab_cols
]
cohort = (
    cohort[required_nonfeature + all_feature_cols]
    .dropna(subset=drop_subset)
    .drop_duplicates(subset='DFCI_MRN', keep='first')
    .reset_index(drop=True)
)

constant_cols = {
    col for col in all_feature_cols if cohort[col].nunique(dropna=False) <= 1
}
if constant_cols:
    cohort = cohort.drop(columns=sorted(constant_cols))
    feature_sets = {
        modality: [col for col in cols if col not in constant_cols]
        for modality, cols in feature_sets.items()
    }

cohort_summary = pd.DataFrame({
    'patients': [len(cohort)],
    'deaths': [int(cohort['death'].sum())],
    'censored': [int((1 - cohort['death']).sum())],
    **{f'n_features_{m}': [len(feature_sets[m])] for m in MODALITIES},
})
display(cohort_summary.T)

## Leakage-safe preprocessing

All imputation is fitted within each training fold. As in the existing Cox implementation, baseline and cancer-type variables are unpenalized, only configured continuous columns are standardized, and PRS features are standardized and reduced to at most 1,500 randomized principal components within each fold. Validation/test records are transformed with that fold's fitted preprocessing objects.

In [ ]:
unpenalized_cols = unique_in_order(baseline_cols)
preprocessing_by_feature = {}
for modality in MODALITIES:
    cols = feature_sets[modality]
    continuous_cols = unique_in_order(modality_cfg[modality]['continuous_vars'])
    preprocessing_by_feature[modality] = {
        'impute_features': True,
        'continuous_indices': [cols.index(col) for col in continuous_cols if col in cols],
        'unpenalized_indices': [cols.index(col) for col in unpenalized_cols if col in cols],
    }
prs_cols = [
    col for col in modality_cfg['prs']['penalized_cols']
    if col in feature_sets['prs']
]
preprocessing_by_feature['prs'].update({
    'pca_indices': [feature_sets['prs'].index(col) for col in prs_cols],
    'pca_components': 1500,
})
{m: len(feature_sets[m]) for m in MODALITIES}

## Fixed 80/20 split, training-only tuning, and test evaluation

In [ ]:
split_assignment = create_train_test_split(
    cohort,
    test_size=0.20,
    random_state=RANDOM_STATE,
)
split_check = (
    cohort[['DFCI_MRN', 'death']]
    .merge(split_assignment, on='DFCI_MRN', validate='one_to_one')
    .groupby('split')['death']
    .agg(['size', 'sum', 'mean'])
)
split_check

In [ ]:
summary, best_hyperparameters = run_train_test_benchmark(
    cohort,
    feature_sets,
    split_assignment,
    OUTPUT_DIR,
    param_grids=PARAM_GRIDS,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    model_names=MODELS,
    feature_set_names=MODALITIES,
    preprocessing_by_feature=preprocessing_by_feature,
)
summary.sort_values(['model', 'mean_auc_t'], ascending=[True, False])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric, title in zip(
    axes,
    ['mean_auc_t', 'c_index', 'integrated_brier_score'],
    ['Mean AUC(t)', 'C-index', 'Integrated Brier score'],
):
    summary.pivot(index='feature_set', columns='model', values=metric).plot.bar(ax=ax)
    ax.set_title(title)
    ax.set_xlabel('Modality')
plt.tight_layout();

## Whole-cohort five-fold held-out modality risk scores

Using the selected parameters, cross-fit all three algorithms across the entire cohort. The wide output contains 18 held-out columns (`<model>__<modality>`), and a separate file is also written for each model–modality pair.

In [ ]:
risk_scores, oof_timing = generate_oof_risk_scores(
    cohort,
    feature_sets,
    best_hyperparameters,
    OUTPUT_DIR,
    n_splits=5,
    random_state=RANDOM_STATE,
    n_jobs=N_JOBS,
    model_names=MODELS,
    feature_set_names=MODALITIES,
    preprocessing_by_feature=preprocessing_by_feature,
)
risk_cols = [col for col in risk_scores if '__' in col]
assert len(risk_cols) == len(MODELS) * len(MODALITIES)
assert risk_scores[risk_cols].notna().all().all()
display(risk_scores.head())
display(oof_timing.query("fold == 'all'").sort_values(['model', 'feature_set']))

In [ ]:
risk_correlations = risk_scores[risk_cols].corr()
risk_correlations.to_csv(OUTPUT_DIR / 'oof_risk_score_correlations.csv')
print(f'Performance and timing: {OUTPUT_DIR / "test_performance_and_timing.csv"}')
print(f'Held-out scores: {OUTPUT_DIR / "mortality_oof_risk_scores.csv"}')